# kin_08 — How the nose and jaw move

What do `nose_tip` and `jaw` do over a session, how well are they tracked, and what is noise
in their positions? The answers set the defaults for processing these two keypoints with the
library's kinematics core (likelihood threshold, smoothing cutoff, speed thresholds for
segmenting movements) and the per-session quality numbers. Matching their movements to the task
comes after.

Everything here is computed from the raw pose (`pose.parquet` of the library batch), before any
masking or smoothing, except §8, which runs the library's `signal.smooth`.

**§1 runs on Code Ocean only:** it ranks every session in the batch by how well nose and jaw
are tracked, picks example sessions, and writes a small copy of them (nose, jaw and tongue pose,
licks, trials, record) to `scratch/nose_jaw_sessions/` for local work. Locally, put those folders
in `data/for_local/nose_jaw_sessions/` (or point `NOSE_JAW_ROOT` at them); §2–§10 then run on
them. §6 (video frames) needs the videos, so Code Ocean only.

## Setup

In [ ]:
%matplotlib inline
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal as sps

from kin_utils import KINEMATICS_ROOT
from signal_utils import near_any
from plotstyle import apply_style, OKABE_ITO, PALETTE, style_ax, save_fig
apply_style()

In [ ]:
IS_CO = Path("/root/capsule").exists()
REPO = Path.cwd().parent                                   # notebooks run from code/
if IS_CO:
    SESSION_ROOT = KINEMATICS_ROOT
    SCRATCH = Path("/root/capsule/scratch")
    DATA_ROOT = Path("/root/capsule/data")
else:
    SESSION_ROOT = Path(os.environ.get("NOSE_JAW_ROOT",
                                       REPO / "data" / "for_local" / "nose_jaw_sessions"))
    SCRATCH = REPO / "data" / "for_local"
EXPORT_ROOT = SCRATCH / "nose_jaw_sessions"                # §1 writes the example sessions here
FIG_DIR = SCRATCH / "figures" / "kin_08_nose_jaw_movement"
SAVE_FIG = False

NOSE, JAW, TONGUE = "nose_tip", "jaw", "tongue_tip_center"
KEYPOINTS = (NOSE, JAW)
COLORS = {NOSE: OKABE_ITO["reddish_purple"], JAW: OKABE_ITO["bluish_green"]}
CONTEXT_COLORS = {"licking": OKABE_ITO["orange"], "no licking": PALETTE["neutral"]}
FS = 500.0                     # bottom camera frame rate
CONF = 0.9                     # likelihood threshold for "confident" (as for the tongue)
LICKING_S = 0.25               # a frame is "licking" within this of a lick ...
NO_LICKING_S = 1.0             # ... and "no licking" when no lick is within this
print("Code Ocean" if IS_CO else "local", "| sessions in", SESSION_ROOT)

In [ ]:
def read_pose(session_dir, keypoints=KEYPOINTS):
    # Raw pose of some keypoints, one row per frame per keypoint, session time.
    pose = pd.read_parquet(Path(session_dir) / "pose.parquet",
                           columns=["frame", "time", "keypoint", "x", "y", "likelihood"],
                           filters=[("keypoint", "in", list(keypoints))])
    pose["keypoint"] = pose["keypoint"].astype(str)
    return pose.sort_values(["keypoint", "frame"]).reset_index(drop=True)


def read_record(session_dir):
    return json.loads((Path(session_dir) / "record.json").read_text())


def steps(g):
    # Frame-to-frame displacement (px) and whether both frames are confident; index = later frame.
    step = np.hypot(np.diff(g["x"].to_numpy(float)), np.diff(g["y"].to_numpy(float)))
    ok = g["likelihood"].to_numpy(float) >= CONF
    return step, ok[1:] & ok[:-1]

## 1. Sessions ranked by nose and jaw tracking (Code Ocean)

Per session and keypoint: fraction of frames with likelihood ≥ `CONF`, and the median and
99.9th percentile of frame-to-frame steps between confident frames (the first is close to the
position jitter, the second catches jumps). Sessions are ranked by the lower of the two
confident fractions. Cached in `scratch/kin_08_session_tracking.csv`; delete it to recompute.

In [ ]:
def tracking_summary(session_dir):
    rec = read_record(session_dir)
    pose = read_pose(session_dir)
    row = {"session": Path(session_dir).name,
           "timing": rec.get("timing", {}).get("verdict"),
           "lick_coverage": (rec.get("quality") or {}).get("lick_coverage", np.nan),
           "minutes": (pose["time"].max() - pose["time"].min()) / 60}
    for kp, g in pose.groupby("keypoint"):
        step, both = steps(g)
        row[f"{kp}_confident"] = float((g["likelihood"] >= CONF).mean())
        row[f"{kp}_step_p50"] = float(np.nanmedian(step[both])) if both.any() else np.nan
        row[f"{kp}_step_p999"] = (float(np.nanpercentile(step[both], 99.9))
                                  if both.any() else np.nan)
    return row


TRACKING_CSV = SCRATCH / "kin_08_session_tracking.csv"
if IS_CO:
    if TRACKING_CSV.exists():
        tracking = pd.read_csv(TRACKING_CSV)
    else:
        dirs = sorted(p.parent for p in SESSION_ROOT.glob("*/record.json"))
        rows = []
        for i, d in enumerate(dirs):
            try:
                rows.append(tracking_summary(d))
            except Exception as e:                       # report and go on
                print(d.name, type(e).__name__, e)
            if i % 20 == 0:
                print(f"{i + 1}/{len(dirs)}", flush=True)
        tracking = pd.DataFrame(rows)
        tracking.to_csv(TRACKING_CSV, index=False)
    tracking["score"] = tracking[[f"{k}_confident" for k in KEYPOINTS]].min(axis=1)
    tracking = tracking.sort_values("score", ascending=False).reset_index(drop=True)
    print(len(tracking), "sessions")
    display(tracking.head(10).round(3))
    display(tracking.tail(5).round(3))

In [ ]:
if IS_CO:
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
    ax = axes[0]
    ax.scatter(tracking[f"{NOSE}_confident"], tracking[f"{JAW}_confident"], s=12,
               color=PALETTE["all"])
    ax.set_xlabel(f"{NOSE}: fraction confident"); ax.set_ylabel(f"{JAW}: fraction confident")
    for ax, stat, label in ((axes[1], "step_p50", "median step (px)"),
                            (axes[2], "step_p999", "99.9th pct step (px)")):
        for kp in KEYPOINTS:
            ax.hist(tracking[f"{kp}_{stat}"].dropna(), bins=40, alpha=0.6, color=COLORS[kp],
                    label=kp)
        ax.set_xlabel(label); ax.set_ylabel("sessions")
    axes[1].legend(frameon=False)
    for ax in axes:
        style_ax(ax)
    fig.tight_layout()
    plt.show()

Example sessions: the best tracked, two near the median, and the worst. Their nose, jaw and
tongue pose (float32), licks, trials and `record.json` are written to `EXPORT_ROOT`, about
50–100 MB a session; download that folder for local work.

In [ ]:
if IS_CO:
    mid = len(tracking) // 2
    EXAMPLES = list(dict.fromkeys(
        [tracking.session.iloc[0], tracking.session.iloc[mid], tracking.session.iloc[mid + 1],
         tracking.session.iloc[-1]]))
    for s in EXAMPLES:
        out = EXPORT_ROOT / s
        if (out / "record.json").exists():
            continue
        out.mkdir(parents=True, exist_ok=True)
        p = read_pose(SESSION_ROOT / s, KEYPOINTS + (TONGUE,))
        p[["x", "y", "likelihood"]] = p[["x", "y", "likelihood"]].astype("float32")
        p.to_parquet(out / "pose.parquet", index=False)
        for name in ("licks", "trials"):
            pd.read_parquet(SESSION_ROOT / s / f"{name}.parquet").to_parquet(
                out / f"{name}.parquet", index=False)
        (out / "record.json").write_text((SESSION_ROOT / s / "record.json").read_text())
    print("examples:", EXAMPLES, "->", EXPORT_ROOT)

## 2. One session: raw positions

Change `SESSION` and rerun from here to look at another one. Positions are in image pixels
minus the keypoint's session median (confident frames), so 0 is where the point usually is.

In [ ]:
SESSIONS = sorted(p.parent.name for p in SESSION_ROOT.glob("*/record.json"))
SESSION = EXAMPLES[0] if IS_CO else SESSIONS[0]           # or any name in SESSIONS
SESSION_DIR = SESSION_ROOT / SESSION

pose = read_pose(SESSION_DIR)
record = read_record(SESSION_DIR)
licks = pd.read_parquet(SESSION_DIR / "licks.parquet")
trials = pd.read_parquet(SESSION_DIR / "trials.parquet")
lick_t = np.sort(licks["timestamps"].dropna().to_numpy(float))
go_t = np.sort(trials["goCue_start_time_in_session"].dropna().to_numpy(float))

# One frame-indexed table per keypoint, with positions relative to the session median.
kp = {}
for name, g in pose.groupby("keypoint"):
    g = g.set_index("frame")
    g["conf"] = g["likelihood"] >= CONF
    g["dx"] = g["x"] - g.loc[g.conf, "x"].median()
    g["dy"] = g["y"] - g.loc[g.conf, "y"].median()
    kp[name] = g
t = kp[JAW]["time"].to_numpy(float)

# Lick context per frame
licking = near_any(t, lick_t, -LICKING_S, LICKING_S)
no_licking = ~near_any(t, lick_t, -NO_LICKING_S, NO_LICKING_S)
CONTEXTS = {"licking": licking, "no licking": no_licking}

print(SESSION, f"| {len(t):,} frames, {(t[-1] - t[0]) / 60:.1f} min | {len(lick_t)} licks, "
      f"{len(go_t)} go cues | timing {record.get('timing', {}).get('verdict')}")
print({c: f"{m.mean():.0%} of frames" for c, m in CONTEXTS.items()})

In [ ]:
def plot_window(t0, t1):
    # Raw x, y (relative to median) and likelihood; low-likelihood frames in grey.
    sel = (t >= t0) & (t <= t1)
    fig, axes = plt.subplots(5, 1, figsize=(11, 8), sharex=True)
    rows = [(NOSE, "dx"), (NOSE, "dy"), (JAW, "dx"), (JAW, "dy")]
    for ax, (name, col) in zip(axes, rows):
        g = kp[name][sel]
        ax.plot(g["time"], g[col].where(g.conf), lw=0.7, color=COLORS[name])
        ax.plot(g["time"], g[col].where(~g.conf), ".", ms=2, color=PALETTE["not_sig"])
        ax.set_ylabel(f"{name}\n{col[1]} (px)")
    for name in KEYPOINTS:
        g = kp[name][sel]
        axes[4].plot(g["time"], g["likelihood"], lw=0.7, color=COLORS[name], label=name)
    axes[4].axhline(CONF, color=PALETTE["neutral"], lw=0.5, ls="--")
    axes[4].set_ylabel("likelihood"); axes[4].legend(frameon=False, ncol=2)
    for ax in axes:
        for x in lick_t[(lick_t >= t0) & (lick_t <= t1)]:
            ax.axvline(x, color=CONTEXT_COLORS["licking"], lw=0.4, alpha=0.6)
        for x in go_t[(go_t >= t0) & (go_t <= t1)]:
            ax.axvline(x, color="k", lw=0.8)
        style_ax(ax)
    axes[-1].set_xlabel("session time (s)  |  black: go cue, orange: lick")
    fig.tight_layout()
    return fig


# A go cue in the middle of the session that was followed by a lick
answered = go_t[near_any(go_t, lick_t, 0.0, 1.0)]
GO_EXAMPLE = answered[len(answered) // 2]
plot_window(GO_EXAMPLE - 5, GO_EXAMPLE + 15)
plt.show()

In [ ]:
# A 2-s zoom on the start of that response
plot_window(GO_EXAMPLE - 0.5, GO_EXAMPLE + 1.5)
plt.show()

Over the whole session: median position (confident frames) and fraction of confident frames in
10-s bins. A head-fixed animal's nose and jaw should stay put; steps or drift here mean the
animal or camera shifted, which a single session-median landmark would not capture.

In [ ]:
bins = np.arange(t[0], t[-1] + 10, 10.0)
fig, axes = plt.subplots(3, 1, figsize=(11, 6), sharex=True)
for name in KEYPOINTS:
    g = kp[name]
    b = pd.cut(g["time"], bins)
    med = g[g.conf].groupby(b[g.conf], observed=False)[["dx", "dy"]].median()
    centre = bins[:-1] + 5
    axes[0].plot(centre, med["dx"], color=COLORS[name], label=name)
    axes[1].plot(centre, med["dy"], color=COLORS[name])
    axes[2].plot(centre, g.groupby(b, observed=False)["conf"].mean(), color=COLORS[name])
axes[0].set_ylabel("median x (px)"); axes[1].set_ylabel("median y (px)")
axes[2].set_ylabel("fraction confident"); axes[2].set_xlabel("session time (s)")
axes[0].legend(frameon=False)
for ax in axes:
    style_ax(ax)
fig.tight_layout()
plt.show()

## 3. Likelihood

Distribution of the tracker's likelihood per keypoint, overall and by lick context, and the
fraction of frames above a few thresholds.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
edges = np.linspace(0, 1, 51)
for ax, name in zip(axes, KEYPOINTS):
    lik = kp[name]["likelihood"].to_numpy()
    for ctx, m in CONTEXTS.items():
        ax.hist(lik[m], bins=edges, density=True, histtype="step", lw=1.2,
                color=CONTEXT_COLORS[ctx], label=ctx)
    ax.axvline(CONF, color=PALETTE["neutral"], lw=0.5, ls="--")
    ax.set_yscale("log"); ax.set_title(name); ax.set_xlabel("likelihood")
    style_ax(ax)
axes[0].set_ylabel("density"); axes[0].legend(frameon=False)
fig.tight_layout()
plt.show()

thresholds = [0.5, 0.8, 0.9, 0.95, 0.99]
display(pd.DataFrame({name: [(kp[name]["likelihood"] >= th).mean() for th in thresholds]
                      for name in KEYPOINTS}, index=[f"≥ {th}" for th in thresholds]).round(4))

In [ ]:
# Lengths of low-likelihood stretches: isolated frames or long losses?
def run_lengths(mask):
    m = np.r_[False, mask, False].astype(int)
    d = np.diff(m)
    return np.flatnonzero(d == -1) - np.flatnonzero(d == 1)

rows = []
for name in KEYPOINTS:
    runs = run_lengths(~kp[name]["conf"].to_numpy())
    rows.append({"keypoint": name, "stretches": len(runs),
                 **{f"{q:.0%} (frames)": np.percentile(runs, q * 100) if len(runs) else np.nan
                    for q in (0.5, 0.9, 0.99)},
                 "longest (s)": runs.max() / FS if len(runs) else 0.0})
display(pd.DataFrame(rows).set_index("keypoint").round(2))

## 4. Where the points are

2-D histogram of confident positions (log counts), with low-likelihood positions on top in
grey. A second cluster of confident points would be a confident misdetection of another body
part. The principal axes of the confident positions give the direction each point moves most
and its spread along and across it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes_rows = []
for ax, name in zip(axes, KEYPOINTS):
    g = kp[name]
    c = g[g.conf]
    lim = np.nanpercentile(np.abs(np.r_[c.dx, c.dy]), 99.9) * 1.5
    ax.hist2d(c.dx, c.dy, bins=120, range=[[-lim, lim], [-lim, lim]], cmap="Greys",
              norm=plt.matplotlib.colors.LogNorm())
    low = g[~g.conf]
    ax.plot(low.dx, low.dy, ".", ms=1, alpha=0.3, color=COLORS[name])
    ax.set_xlim(-lim, lim); ax.set_ylim(lim, -lim)           # image y points down
    ax.set_aspect("equal"); ax.set_title(name)
    ax.set_xlabel("x − median (px)"); ax.set_ylabel("y − median (px)")
    # principal axes
    xy = c[["dx", "dy"]].to_numpy()
    evals, evecs = np.linalg.eigh(np.cov(xy.T))
    major = evecs[:, 1]
    for v, s in zip(evecs.T, np.sqrt(evals)):
        ax.plot([-3 * s * v[0], 3 * s * v[0]], [-3 * s * v[1], 3 * s * v[1]], color=COLORS[name])
    axes_rows.append({"keypoint": name, "SD major (px)": np.sqrt(evals[1]),
                      "SD minor (px)": np.sqrt(evals[0]),
                      "major axis angle (deg from +x)": np.degrees(np.arctan2(major[1], major[0])) % 180,
                      "SD x (px)": c.dx.std(), "SD y (px)": c.dy.std()})
    style_ax(ax)
fig.tight_layout()
plt.show()
display(pd.DataFrame(axes_rows).set_index("keypoint").round(2))

## 5. Frame-to-frame steps

Displacement between consecutive confident frames, by lick context. At 500 Hz, real movement
moves a point a fraction of a pixel to a few pixels per frame; the bottom of the distribution is
the tracker's jitter and the far tail is jumps. The second panel asks whether low likelihood
goes with large steps (median step per likelihood bin, either frame of the pair).

In [ ]:
step = {name: steps(kp[name]) for name in KEYPOINTS}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
edges = np.logspace(-2.5, 2.5, 80)
for ax, name in zip(axes[:2], KEYPOINTS):
    s, both = step[name]
    for ctx, m in CONTEXTS.items():
        sel = both & m[1:]
        ax.hist(s[sel], bins=edges, density=True, histtype="step", lw=1.2,
                color=CONTEXT_COLORS[ctx], label=f"{ctx} ({sel.sum():,})")
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_title(name)
    ax.set_xlabel("step between confident frames (px)"); ax.legend(frameon=False, fontsize=8)
    style_ax(ax)
axes[0].set_ylabel("density")

ax = axes[2]
lik_edges = np.linspace(0, 1, 21)
for name in KEYPOINTS:
    s, _ = step[name]
    lik = kp[name]["likelihood"].to_numpy()
    pair_lik = np.minimum(lik[1:], lik[:-1])
    b = np.digitize(pair_lik, lik_edges) - 1
    med = pd.Series(s).groupby(b).median().reindex(range(len(lik_edges) - 1))
    ax.plot(lik_edges[:-1] + 0.025, med.to_numpy(), "o-", ms=3, color=COLORS[name], label=name)
ax.set_yscale("log"); ax.set_xlabel("lower likelihood of the pair")
ax.set_ylabel("median step (px)"); ax.legend(frameon=False)
style_ax(ax)
fig.tight_layout()
plt.show()

display(pd.DataFrame({name: pd.Series(step[name][0][step[name][1]]).describe(
    percentiles=[0.5, 0.9, 0.99, 0.999]) for name in KEYPOINTS}).round(3))

## 6. Frames from the video (Code Ocean)

For each keypoint: random confident frames, the lowest-likelihood frames, and the frames after
the largest confident steps. Crosses are the tracked positions (nose, jaw), drawn on the raw
bottom-camera frame cropped around the face. Frames are found by index
(`video_clips.write_frames`), so dropped frames do not shift them.

In [ ]:
N_FRAMES = 6
rng = np.random.default_rng(0)


def frames_to_check(name):
    g = kp[name]
    s, both = step[name]
    frame = g.index.to_numpy()
    conf_frames = frame[g.conf.to_numpy()]
    big = np.argsort(np.where(both, s, -np.inf))[::-1][:N_FRAMES] + 1   # later frame of the pair
    return {"random confident": np.sort(rng.choice(conf_frames, N_FRAMES, replace=False)),
            "lowest likelihood": frame[np.argsort(g["likelihood"].to_numpy())[:N_FRAMES]],
            "largest steps": frame[big]}


CHECK = {name: frames_to_check(name) for name in KEYPOINTS}
CHECK

In [ ]:
def source_video(record, session):
    # Raw MP4 next to the camera's video CSV; else the Lightning Pose labeled video.
    csv = Path(record["video_csv"])
    candidates = [csv.with_suffix(".mp4")]
    if csv.name == "metadata.csv":                         # AIND layout: one camera per folder
        candidates += sorted(csv.parent.glob("*.mp4"))
    for c in candidates:
        if c.exists():
            return c
    from aind_dynamic_foraging_behavior_video_analysis.kinematics.tasks import (
        dynamic_foraging_tongue as dft)
    return Path(dft.find_labeled_video(session, DATA_ROOT))


def show_frames(name, kind, frames, pngs, half=120):
    centre = np.nanmean([[kp[k].loc[kp[k].conf, "x"].median(), kp[k].loc[kp[k].conf, "y"].median()]
                         for k in KEYPOINTS], axis=0)
    fig, axes = plt.subplots(1, len(frames), figsize=(2.6 * len(frames), 2.9))
    for ax, f, png in zip(np.atleast_1d(axes), frames, pngs):
        img = plt.imread(png)
        ax.imshow(img, cmap="gray")
        for k in KEYPOINTS:
            r = kp[k].loc[f]
            ax.plot(r.x, r.y, "+", ms=12, mew=1.5, color=COLORS[k])
        ax.set_xlim(centre[0] - half, centre[0] + half)
        ax.set_ylim(centre[1] + half, centre[1] - half)
        ax.set_title(f"f{f}  {name[:4]} L={kp[name].loc[f, 'likelihood']:.2f}", fontsize=8)
        ax.axis("off")
    fig.suptitle(f"{name}: {kind}", fontsize=10)
    fig.tight_layout()
    plt.show()


if IS_CO:
    from aind_dynamic_foraging_behavior_video_analysis import video_clips
    VIDEO = source_video(record, SESSION)
    PNG_DIR = SCRATCH / "kin_08_frames" / SESSION
    print("video:", VIDEO)
    for name, kinds in CHECK.items():
        for kind, frames in kinds.items():
            pngs = video_clips.write_frames(VIDEO, frames, PNG_DIR, prefix=SESSION)
            show_frames(name, kind, frames, pngs)

## 7. Spectra

Power spectra of raw x and y, from 1-s windows wholly in one lick context, without dropped frames,
with at least 95% of frames confident and no low-likelihood stretch over 5 frames (filled
linearly). Where the licking spectrum meets the no-licking one at high
frequencies, and where both flatten, is the noise floor; the smoothing cutoff belongs below it.
Dashed lines: 30 Hz (the sandbox's jaw cutoff) and 50 Hz (the tongue's).

In [ ]:
NPER = 512                                                   # ~1 s at 500 Hz
MIN_OK, MAX_FILL = 0.95, 5
steady = np.r_[True, np.diff(t) < 1.5 / FS]                   # no dropped frame before this one


def window_psd(values, ok, context):
    # Mean Welch spectrum over 1-s windows wholly in one context, without dropped frames, with at
    # least MIN_OK of frames confident and no low-likelihood stretch longer than MAX_FILL frames
    # (filled linearly).
    n = len(values) // NPER
    v = values[: n * NPER].reshape(n, NPER)
    good = ok[: n * NPER].reshape(n, NPER)
    usable = context[: n * NPER].reshape(n, NPER).all(axis=1) & (good.mean(axis=1) >= MIN_OK)
    segs = []
    for w in np.flatnonzero(usable):
        if run_lengths(~good[w]).max(initial=0) > MAX_FILL or not good[w, [0, -1]].all():
            continue
        k = np.arange(NPER)
        segs.append(np.interp(k, k[good[w]], v[w, good[w]]))
    if not segs:
        return None, None, 0
    f, p = sps.welch(np.array(segs), fs=FS, nperseg=NPER, axis=1)
    return f, p.mean(axis=0), len(segs)


fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for i, name in enumerate(KEYPOINTS):
    g = kp[name]
    for j, col in enumerate(("x", "y")):
        ax = axes[i, j]
        for ctx, m in CONTEXTS.items():
            f, p, n = window_psd(g[col].to_numpy(float), g.conf.to_numpy(), m & steady)
            if f is not None:
                ax.loglog(f[1:], p[1:], color=CONTEXT_COLORS[ctx], label=f"{ctx} ({n} s)")
        for hz in (30, 50):
            ax.axvline(hz, color=PALETTE["neutral"], lw=0.5, ls="--")
        ax.set_title(f"{name} {col}"); ax.set_ylabel("px² / Hz")
        ax.legend(frameon=False, fontsize=8)
        style_ax(ax)
for ax in axes[-1]:
    ax.set_xlabel("Hz")
fig.tight_layout()
plt.show()

## 8. Speed after smoothing

Each keypoint masked at `CONF` and smoothed with the library's `signal.smooth` (per tracked run,
zero-phase Butterworth) at several cutoffs. The speed distribution by lick context shows which
cutoff separates still from moving; the no-licking distribution at low speed is the jitter
that a segmentation threshold has to sit above.

In [ ]:
from aind_dynamic_foraging_behavior_video_analysis.kinematics import signal as ksig

CUTOFFS = [None, 50.0, 30.0, 20.0, 10.0]
smoothed = {}
for name in KEYPOINTS:
    frames = ksig.mask(pose[pose.keypoint == name].reset_index(drop=True), CONF)
    for c in CUTOFFS:
        smoothed[name, c] = ksig.smooth(frames, FS, cutoff_hz=c)

fig, axes = plt.subplots(2, len(CUTOFFS), figsize=(3 * len(CUTOFFS), 6), sharex=True, sharey="row")
edges = np.logspace(-0.5, 4, 80)
rows = []
for i, name in enumerate(KEYPOINTS):
    for j, c in enumerate(CUTOFFS):
        sp = smoothed[name, c]["speed"].to_numpy()
        ax = axes[i, j]
        for ctx, m in CONTEXTS.items():
            ax.hist(sp[m], bins=edges, density=True, histtype="step", lw=1.2,
                    color=CONTEXT_COLORS[ctx], label=ctx)
            q = np.nanpercentile(sp[m], [50, 90, 99])
            rows.append({"keypoint": name, "cutoff (Hz)": c or "none", "context": ctx,
                         "p50": q[0], "p90": q[1], "p99": q[2]})
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_title(f"{name}, {c or 'no'} Hz" if c else f"{name}, unsmoothed", fontsize=9)
        style_ax(ax)
    axes[i, 0].set_ylabel("density")
for ax in axes[-1]:
    ax.set_xlabel("speed (px/s)")
axes[0, 0].legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()
display(pd.DataFrame(rows).pivot_table(index=["keypoint", "cutoff (Hz)"], columns="context",
                                       values=["p50", "p90", "p99"]).round(1))

In [ ]:
# The same 2-s zoom as §2, smoothed at each cutoff
t0, t1 = GO_EXAMPLE - 0.5, GO_EXAMPLE + 1.5
fig, axes = plt.subplots(2, 2, figsize=(11, 5), sharex=True)
shades = plt.cm.viridis(np.linspace(0, 0.9, len(CUTOFFS)))
for i, name in enumerate(KEYPOINTS):
    for j, col in enumerate(("x", "y")):
        ax = axes[i, j]
        for c, colr in zip(CUTOFFS, shades):
            s = smoothed[name, c]
            sel = (s.time >= t0) & (s.time <= t1)
            ax.plot(s.time[sel], s[col][sel], lw=0.8, color=colr, label=f"{c or 'none'} Hz")
        for x in lick_t[(lick_t >= t0) & (lick_t <= t1)]:
            ax.axvline(x, color=CONTEXT_COLORS["licking"], lw=0.4)
        ax.set_title(f"{name} {col}"); ax.set_ylabel("px")
        style_ax(ax)
axes[0, 0].legend(frameon=False, fontsize=7, ncol=3)
fig.tight_layout()
plt.show()

## 9. Jaw relative to nose

In a head-fixed mouse, the jaw relative to the nose separates mouth opening from movement of
the whole head or face (and from camera shifts). Frames with both confident. Left: the jaw's
position relative to the nose; right: the nose–jaw distance by lick context.

In [ ]:
both_conf = kp[NOSE].conf.to_numpy() & kp[JAW].conf.to_numpy()
rel_x = (kp[JAW]["x"] - kp[NOSE]["x"]).to_numpy()
rel_y = (kp[JAW]["y"] - kp[NOSE]["y"]).to_numpy()
dist = np.hypot(rel_x, rel_y)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax = axes[0]
ax.hist2d(rel_x[both_conf], rel_y[both_conf], bins=120, cmap="Greys",
          norm=plt.matplotlib.colors.LogNorm())
ax.invert_yaxis(); ax.set_aspect("equal")
ax.set_xlabel("jaw x − nose x (px)"); ax.set_ylabel("jaw y − nose y (px)")
style_ax(ax)
ax = axes[1]
for ctx, m in CONTEXTS.items():
    ax.hist(dist[both_conf & m], bins=100, density=True, histtype="step", lw=1.2,
            color=CONTEXT_COLORS[ctx], label=ctx)
ax.set_xlabel("nose–jaw distance (px)"); ax.set_ylabel("density"); ax.legend(frameon=False)
style_ax(ax)
fig.tight_layout()
plt.show()
print(f"both confident: {both_conf.mean():.1%} of frames; distance median "
      f"{np.median(dist[both_conf]):.1f} px, IQR {np.subtract(*np.percentile(dist[both_conf], [75, 25])):.1f} px")

## 10. Nose and jaw around a lick

Mean raw position (relative to the session median, confident frames only) from 0.3 s before to
0.3 s after each lick, ± SD across licks, with the jaw relative to the nose in the last column.
Licking is the jaw's most frequent movement, so this is its basic movement cycle.

In [ ]:
lags = np.arange(-0.3, 0.3 + 1 / FS, 1 / FS)
idx = np.clip(np.searchsorted(t, lick_t[:, None] + lags[None, :]), 0, len(t) - 1)
off = np.abs(t[idx] - (lick_t[:, None] + lags[None, :])) > 1.5 / FS   # dropped frames / ends

traces = {}
for name in KEYPOINTS:
    g = kp[name]
    for col in ("dx", "dy"):
        v = g[col].where(g.conf).to_numpy(float)[idx]
        v[off] = np.nan
        traces[name, col] = v
for col, rel in (("dx", rel_x), ("dy", rel_y)):
    v = np.where(both_conf, rel, np.nan)[idx]
    v[off] = np.nan
    traces["jaw − nose", col] = v - np.nanmedian(rel[both_conf])

fig, axes = plt.subplots(2, 3, figsize=(12, 5), sharex=True)
for j, name in enumerate([NOSE, JAW, "jaw − nose"]):
    for i, col in enumerate(("dx", "dy")):
        v = traces[name, col]
        m, s = np.nanmean(v, axis=0), np.nanstd(v, axis=0)
        colr = COLORS.get(name, PALETTE["all"])
        ax = axes[i, j]
        ax.fill_between(lags, m - s, m + s, color=colr, alpha=0.2, lw=0)
        ax.plot(lags, m, color=colr)
        ax.axvline(0, color=CONTEXT_COLORS["licking"], lw=0.8)
        ax.set_title(f"{name} {col[1]}", fontsize=9)
        style_ax(ax)
    axes[1, j].set_xlabel("time from lick (s)")
axes[0, 0].set_ylabel("px"); axes[1, 0].set_ylabel("px")
fig.tight_layout()
plt.show()
print(len(lick_t), "licks")

## Next

Run §2–§10 on the other `EXAMPLES` (change `SESSION`), then turn the numbers that separate
good from bad tracking into per-session measures over every session, as §1 does for the
confident fraction and steps.